# Case Studies (Phase 6)

Inspect qualitative differences between standard and fairness-aware KG2RAG outputs.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
ROOT = Path.cwd().parent
pred_dir = ROOT / 'outputs' / 'predictions'
split = 'dev'

standard_path = pred_dir / f'kg2rag_standard_generation_{split}.json'
fair_path = pred_dir / f'kg2rag_fair_generation_{split}.json'

if not standard_path.exists() or not fair_path.exists():
    print('Missing generation artifacts. Run experiments first.')
else:
    std_df = pd.read_json(standard_path)
    fair_df = pd.read_json(fair_path)

    merged = std_df.merge(
        fair_df,
        on='id',
        suffixes=('_standard', '_fair'),
    )
    merged['answer_changed'] = merged['answer_standard'] != merged['answer_fair']
    merged['context_len_standard'] = merged['metadata_standard'].apply(lambda x: x.get('context_length') if isinstance(x, dict) else None)
    merged['context_len_fair'] = merged['metadata_fair'].apply(lambda x: x.get('context_length') if isinstance(x, dict) else None)
    display(merged[['id', 'answer_changed', 'gender_standard', 'geo_group_standard']].head())

In [ ]:
if 'merged' in locals():
    changed = merged[merged['answer_changed']].copy()
    print(f'Cases with answer changes: {len(changed)} / {len(merged)}')
    display(changed[['id', 'question_standard', 'answer_standard', 'answer_fair']].head(10))

In [ ]:
if 'merged' in locals() and not merged.empty:
    sample = merged.sample(min(3, len(merged)), random_state=42)
    for _, row in sample.iterrows():
        print('=' * 80)
        print('ID:', row['id'])
        print('Question:', row['question_standard'])
        print('Gender/Geo:', row.get('gender_standard'), row.get('geo_group_standard'))
        print('Standard answer:', row['answer_standard'])
        print('Fair answer:', row['answer_fair'])
        print('Standard supporting titles:', row.get('predicted_supporting_titles_standard'))
        print('Fair supporting titles:', row.get('predicted_supporting_titles_fair'))